In [264]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier


In [265]:
df = pd.read_csv("premier_league_2025_26_matches.csv")
df2 = pd.read_csv("prem_stats.csv")

team_mapping = {
    "man city": "manchester city",
    "man united": "manchester united",
    "leeds": "leeds united",
    "nott'm forest": "nottingham forest"
}

columns = ["team_name", "opposing_team"]

for col in columns:
    df[col] = df[col].str.lower().replace(team_mapping)

In [266]:
result_mapping = {
    "l": 0,
    "d": 1,
    "w": 2
}

df["win_or_lose"] = df["win_or_lose"].map(result_mapping)

In [267]:
df_train = pd.merge(
    df,
    df2.rename(columns=lambda c: f"home_{c}" if c != "team_name" else c),
    on="team_name",
    how="left"
)

df_train = df_train.merge(
    df2.rename(columns=lambda c: f"away_{c}" if c != "team_name" else c),
    left_on="opposing_team",
    right_on="team_name",
    how="left"
)

df_train = df_train.drop(columns=["team_name_y"])
df_train = df_train.rename(columns={
    "team_name_x": "team_name"
})

In [268]:
X = df_train.drop(columns=[
    "win_or_lose",
    "team_name",
    "opposing_team",
    "goal_difference",
    "date_played"
])

y = df_train["win_or_lose"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [269]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [270]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.3157894736842105


In [271]:
def create_test_match(fixtures):

    result_mapping = {
        0: "Home Loss",
        1: "Draw",
        2: "Home Win"
    }

    for home_team, away_team in fixtures:

        home = df2.loc[df2["team_name"] == home_team].iloc[0]
        away = df2.loc[df2["team_name"] == away_team].iloc[0]

        test_match = pd.DataFrame([{
            **{
                f"home_{col}": home[col]
                for col in df2.columns if col != "team_name"
            },
            **{
                f"away_{col}": away[col]
                for col in df2.columns if col != "team_name"
            }
        }])

        test_match = test_match[X_train.columns]

        prediction = model.predict(test_match)[0]
        probabilities = model.predict_proba(test_match)[0]

        print(f"\n{'=' * 45}")
        print(f"{home_team.title()} vs {away_team.title()}")
        print(f"Prediction: {result_mapping[prediction]}")

        for cls, probability in zip(model.classes_, probabilities):
            print(f"{result_mapping[cls]}: {probability:.2%}")

In [273]:
fixtures = [
    # Original fixtures
    ("newcastle", "everton"),
    ("liverpool", "arsenal"),
    ("manchester united", "chelsea"),
    ("manchester city", "tottenham"),
    ("brighton", "brentford"),
    ("aston villa", "west ham"),
    ("bournemouth", "crystal palace"),
    ("fulham", "wolves"),
    ("leeds united", "nottingham forest"),
    ("sunderland", "burnley"),
    ("aston villa", "fulham"),

    # Additional matchups
    ("arsenal", "manchester city"),
    ("chelsea", "liverpool"),
    ("tottenham", "newcastle"),
    ("everton", "brighton"),
    ("west ham", "manchester united"),
    ("crystal palace", "aston villa"),
    ("wolves", "bournemouth"),
    ("brentford", "fulham"),
    ("burnley", "leeds united"),

    # Reversed matchups
    ("everton", "newcastle"),
    ("arsenal", "liverpool"),
    ("chelsea", "manchester united"),
    ("tottenham", "manchester city"),
    ("west ham", "aston villa"),
    ("crystal palace", "bournemouth"),
    ("wolves", "fulham"),
    ("nottingham forest", "leeds united"),
    ("burnley", "sunderland"),
    ("fulham", "aston villa")
]

create_test_match(fixtures)


Newcastle vs Everton
Prediction: Home Win
Home Loss: 45.05%
Draw: 4.86%
Home Win: 50.10%

Liverpool vs Arsenal
Prediction: Home Win
Home Loss: 24.76%
Draw: 1.48%
Home Win: 73.76%

Manchester United vs Chelsea
Prediction: Home Win
Home Loss: 8.30%
Draw: 3.43%
Home Win: 88.26%

Manchester City vs Tottenham
Prediction: Draw
Home Loss: 35.16%
Draw: 44.29%
Home Win: 20.55%

Brighton vs Brentford
Prediction: Home Win
Home Loss: 6.26%
Draw: 11.11%
Home Win: 82.63%

Aston Villa vs West Ham
Prediction: Home Win
Home Loss: 4.60%
Draw: 3.32%
Home Win: 92.09%

Bournemouth vs Crystal Palace
Prediction: Home Win
Home Loss: 3.71%
Draw: 44.76%
Home Win: 51.53%

Fulham vs Wolves
Prediction: Home Win
Home Loss: 0.71%
Draw: 1.54%
Home Win: 97.75%

Leeds United vs Nottingham Forest
Prediction: Home Loss
Home Loss: 80.93%
Draw: 7.83%
Home Win: 11.24%

Sunderland vs Burnley
Prediction: Home Win
Home Loss: 39.04%
Draw: 7.92%
Home Win: 53.04%

Aston Villa vs Fulham
Prediction: Home Win
Home Loss: 15.58%
Draw